# BGI Spatial Transcriptomics — Variability & EMT Analysis
This notebook:
1. Loads four BGI **tissue** datasets.
2. Summarises overall and tumour‐specific cell‐type composition.
3. Computes gene‐expression variability metrics (`cv`, `nonzero_ratio`).
4. Visualises variability across epithelial / mesenchymal / EMT‐TF gene categories.
5. Runs a **TIC** pseudo‑time analysis (radius sub‑graph) on *B03425E3* and inspects EMT monotonicity.

> **Prerequisites**  
> • Set your **OpenAI key** in `utils.config.API_KEY` (or edit below).


In [1]:
import warnings
warnings.filterwarnings('ignore')

from tic.data import load_bgi_dataset
from tic.metrics import compute_variability
from tic.plotting import plot_variability
from tic.constant import EPITHELIAL_GENES, MESENCHYMAL_GENES, EMT_TF

# ⚠️ Edit here if you keep your key elsewhere
from utils.config import API_KEY

DATA_DIR = '/Users/zhangjiahao/Dataset/BGI'  # ← adjust if needed


In [2]:
import pandas as pd
import numpy as np
from anndata import AnnData
from typing import Optional, List

def describe_gene_statistics(
    adata: AnnData, 
    gene_list: Optional[List[str]] = None
) -> pd.DataFrame:
    """
    Compute basic statistics for genes in an AnnData object.

    Parameters
    ----------
    adata : AnnData
        Annotated data matrix (cells × genes).
    gene_list : list of str, optional
        List of gene names to subset; if None, compute stats for all genes.

    Returns
    -------
    pd.DataFrame
        DataFrame with gene statistics:
        ['mean', 'std', 'nonzero_cells', 'nonzero_fraction']
    """
    if gene_list is not None:
        genes_available = [g for g in gene_list if g in adata.var_names]
        X = adata[:, genes_available].X
        var_names = genes_available
    else:
        X = adata.X
        var_names = adata.var_names

    if not isinstance(X, np.ndarray):
        X = X.toarray()  # for sparse matrix support

    mean = X.mean(axis=0)
    std = X.std(axis=0)
    nonzero_cells = (X > 0).sum(axis=0)
    nonzero_fraction = nonzero_cells / X.shape[0]

    df = pd.DataFrame({
        'gene': var_names,
        'mean': mean,
        'std': std,
        'nonzero_cells': nonzero_cells,
        'nonzero_fraction': nonzero_fraction
    })

    return df.set_index('gene')

## 1. Load datasets

In [ ]:
sample_ids = [ 'C03628C1', 'B03425E1', 'B02804B5', 'B03425E3']
adatas = {}
for sid in sample_ids:
    adata = load_bgi_dataset(
        sid,
        data_type='cellbin',
        data_dir=DATA_DIR,
        normalize=None,
        log=False,
        min_total_counts_ratio=None,
        llm_model_name='openai',
        llm_kwargs={'api_key': API_KEY, 'mirror': True},
    )
    adatas[sid] = adata
    print(f'{sid}:', adata)


In [ ]:
for dataset in sample_ids:
    adata = adatas[dataset]
    df = describe_gene_statistics(adata, gene_list=list(EPITHELIAL_GENES)+list(MESENCHYMAL_GENES)+list(EMT_TF))
    print(f'{dataset}: \n', df)
    df.to_csv(f'../../tutorial/example_out/statistics/{dataset}_gene_statistics.csv')

### 1.1 Cell‑type composition & tumour subsets

In [ ]:
tumour_adatas = {}
for sid, adata in adatas.items():
    vc = adata.obs['cell_type'].value_counts()
    tumour_types = [c for c in vc.index if 'tumor' in c.lower()]
    print(f'\n📊 {sid} - top cell types:\n', vc.head())
    print(f'Tumour-related types ({len(tumour_types)}): {tumour_types}')
    tumour_adatas[sid] = adata[adata.obs['cell_type'].isin(tumour_types)].copy()


## 2. Gene‑expression variability

In [5]:
# Compute CV & non‑zero ratio for each (full vs tumour)
variability_full  = {}
variability_tumor = {}
for sid in sample_ids:
    variability_full[sid]  = compute_variability(adatas[sid], metrics=['cv', 'nonzero_ratio'])
    variability_tumor[sid] = compute_variability(tumour_adatas[sid], metrics=['cv', 'nonzero_ratio'])


### 2.1 Variability plots

In [ ]:
gene_categories = {
    'Epithelial': EPITHELIAL_GENES,
    'Mesenchymal': MESENCHYMAL_GENES,
    'EMT Transcription Factors': EMT_TF,
}

for metric in ['cv', 'nonzero_ratio']:
    plot_variability(
        variability={
            **{sid: variability_full[sid] for sid in sample_ids},
            **{sid + '_tumour': variability_tumor[sid] for sid in sample_ids},
        },
        metric=metric,
        gene_categories=gene_categories,
        title=f'{metric.upper()} across BGI datasets',
        annotate_emt=True,
    )
